<a href="https://colab.research.google.com/github/jenkins-neo-server/musicgen/blob/main/Musicgen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile
from IPython.display import Audio, display

print("Hello Setup")

# 1. Setup Device for Colab (prioritize CUDA, then MPS, then CPU)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"✅ Using device: {device}")

print("Hello Load Model & Processor")

# 2. Load Model & Processor (using a medium model for better quality)
processor = AutoProcessor.from_pretrained("facebook/musicgen-medium")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-medium")
model.to(device)


print("✅ MusicGen (Transformers version) is ready!")

Hello Setup
✅ Using device: cuda
Hello Load Model & Processor


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/8.04G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/996 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_encoder.shared.weight to text_encoder.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-medium
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

✅ MusicGen (Transformers version) is ready!


In [ ]:
# 1. Set your prompt
# prompt = "Lo-fi hip hop beat, chill vibes, professional, minimal percussion, 85 BPM"
prompt = "contemporary classical piano piece in the style of Ludovico Einaudi, minimalist solo piano with simple and elegant melody, slow tempo, melancholic yet hopeful mood, repeating arpeggio patterns that build gradually, soft dynamics with a sense of development over time, intimate feel like a studio recording with natural room reverb[reference:3][reference:4]"

# 2. Prepare inputs
inputs = processor(
    text=[prompt],
    padding=True,
    return_tensors="pt",
).to(device)

# 3. Generate (2048 tokens is roughly 40 seconds at default settings. Experiment with values like 2560 or 3072 to find a balance between length and quality.)
print("Generating music...")
audio_values = model.generate(**inputs, max_new_tokens=1024) # Reverted to 2048 for better quality control.

# 4. Save and Play
sampling_rate = model.config.audio_encoder.sampling_rate
audio_data = audio_values[0, 0].cpu().numpy()

# Save to file
scipy.io.wavfile.write("bgm_output.wav", rate=sampling_rate, data=audio_data)

# Play in Jupyter
display(Audio(audio_data, rate=sampling_rate))

Generating music...


In [ ]:
# 1. Set your prompt
# prompt = "Lo-fi hip hop beat, chill vibes, professional, minimal percussion, 85 BPM"
prompt = "contemporary classical piano piece in the style of Ludovico Einaudi, minimalist solo piano with simple and elegant melody, slow tempo, melancholic yet hopeful mood, repeating arpeggio patterns that build gradually, soft dynamics with a sense of development over time, intimate feel like a studio recording with natural room reverb[reference:3][reference:4]"

# 2. Prepare inputs
inputs = processor(
    text=[prompt],
    padding=True,
    return_tensors="pt",
).to(device)

# 3. Generate (2048 tokens is roughly 40 seconds at default settings. Experiment with values like 2560 or 3072 to find a balance between length and quality.)
print("Generating music...")
audio_values = model.generate(**inputs, max_new_tokens=2048) # Reverted to 2048 for better quality control.

# 4. Save and Play
sampling_rate = model.config.audio_encoder.sampling_rate
audio_data = audio_values[0, 0].cpu().numpy()

# Save to file
scipy.io.wavfile.write("bgm_output.wav", rate=sampling_rate, data=audio_data)

# Play in Jupyter
display(Audio(audio_data, rate=sampling_rate))

Generating music...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive Mounted")

Mounted at /content/drive
✅ Google Drive Mounted


In [ ]:
import shutil
import os

output_filename = "bgm_output.wav"
drive_path = '/content/drive/MyDrive/MusicGen_Outputs/' # You can change this path

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

# Copy the file to Google Drive
shutil.copy(output_filename, drive_path + output_filename)

print(f"✅ {output_filename} saved to {drive_path}")

✅ bgm_output.wav saved to /content/drive/MyDrive/MusicGen_Outputs/


In [ ]:
# ==========================================
# 1. SETUP & INSTALLATIONS
# ==========================================
import random
import os
import json
import shutil
import scipy.io.wavfile
import torch
from typing import Dict, List
from IPython.display import display, Audio
from google.colab import drive
from transformers import AutoProcessor, MusicgenForConditionalGeneration

# Mount Google Drive
drive.mount('/content/drive')

# ==========================================
# 2. PODCAST GENRE PROFILES
# ==========================================
# By organizing into profiles, we prevent conflicting descriptors
# (like "joyful" mixed with "dissonant horror strings").

GENRE_PROFILES = {
    "children_story": {
        "emotion": ["playful", "joyful", "curious", "whimsical", "bouncy"],
        "style": ["light orchestral", "acoustic", "nursery", "uplifting"],
        "instrument": ["glockenspiel", "acoustic guitar", "pizzicato strings", "woodwinds", "soft piano"],
        "energy": ["medium", "bouncy pacing", "light and upbeat"]
    },
    "horror": {
        "emotion": ["terrifying", "suspenseful", "eerie", "ominous", "dread-inducing"],
        "style": ["dark ambient", "cinematic horror", "drone"],
        "instrument": ["dissonant strings", "deep sub-bass", "screeching synths", "out-of-tune piano", "creepy music box"],
        "energy": ["slow and creeping", "high tension", "sudden dynamic shifts"]
    },
    "romance_sensual": {
        "emotion": ["intimate", "passionate", "sensual", "warm", "romantic"],
        "style": ["smooth r&b", "ambient lo-fi", "cinematic romance"],
        "instrument": ["soft rhodes piano", "smooth saxophone", "warm synth pads", "gentle acoustic guitar"],
        "energy": ["slow and calm", "gentle pacing", "steady groove"]
    },
    "fantasy": {
        "emotion": ["epic", "mysterious", "magical", "wondrous", "heroic"],
        "style": ["orchestral", "cinematic", "folk fantasy"],
        "instrument": ["sweeping strings", "harp arpeggios", "celtic flute", "distant brass", "choral pads"],
        "energy": ["building dynamics", "moderate tempo", "grand and swelling"]
    },
    "inspirational": {
        "emotion": ["uplifting", "hopeful", "triumphant", "motivational", "bright"],
        "style": ["corporate", "cinematic", "post-rock"],
        "instrument": ["bright piano", "building string section", "clean electric guitar", "stadium percussion"],
        "energy": ["gradually building", "high energy", "steady and driving"]
    },
    "confession": {
        "emotion": ["melancholic", "reflective", "vulnerable", "regretful", "somber"],
        "style": ["minimalist", "ambient", "sad core"],
        "instrument": ["solo acoustic guitar", "muted piano", "sparse ambient pads", "slow cello"],
        "energy": ["very slow", "minimal and quiet", "gentle pacing"]
    },
    "drama_aitah": {
        "emotion": ["tense", "awkward", "dramatic", "uncertain", "frustrated"],
        "style": ["tension", "minimalist", "reality tv background"],
        "instrument": ["ticking percussion", "staccato strings", "subtle synth pulses", "wurlitzer"],
        "energy": ["moderate tempo", "persistent rhythm", "nervous pacing"]
    },
    "educational": {
        "emotion": ["neutral", "focused", "clear", "engaged", "informative"],
        "style": ["lo-fi hip hop", "corporate tech", "upbeat ambient"],
        "instrument": ["upbeat marimba", "light synth arpeggios", "snappy drum machine", "clean bass"],
        "energy": ["steady rhythm", "background pace", "moderate tempo"]
    },
    "history": {
        "emotion": ["solemn", "grand", "reflective", "nostalgic", "important"],
        "style": ["classical", "orchestral documentary", "period piece"],
        "instrument": ["cello solo", "timpani rolls", "harpsichord", "viola drone"],
        "energy": ["slow and dignified", "measured pacing", "steady"]
    },
    "true_crime": {
        "emotion": ["dark", "investigative", "suspenseful", "gritty", "cold"],
        "style": ["drone", "dark ambient", "noir"],
        "instrument": ["slow pulsing synths", "sparse piano notes", "low frequency drones", "metallic percussion"],
        "energy": ["slow and steady", "minimal rhythm", "brooding pace"]
    }
}

TEXTURES = [
    "soft and atmospheric", "minimal and subtle", "rich and layered",
    "dreamy and ambient", "pulsing and rhythmic", "sparse and empty"
]

MOVEMENTS = [
    "slowly evolving", "steady and consistent", "with subtle variations", "maintaining a constant groove"
]

TEMPLATES = [
    "A {emotion} background score for a {genre} podcast, featuring {instrument}, with {energy} intensity in a {style} style, {texture}, {movement}, no vocals.",
    "An instrumental {style} track conveying a {emotion} mood for a {genre} story, driven by {instrument}, {energy} energy, {texture}, {movement}, no vocals.",
    "A {emotion} background music piece designed for {genre} narration, using {instrument}, with {energy} pacing, {texture}, {movement}, no vocals."
]

# ==========================================
# 3. PROMPT GENERATOR
# ==========================================
def generate_genre_prompts(n_per_genre: int = 10) -> List[Dict]:
    """Generates a perfectly balanced list of prompts across all defined genres."""
    generated_results = []

    for genre, profile in GENRE_PROFILES.items():
        seen_prompts = set()
        attempts = 0

        while len(seen_prompts) < n_per_genre and attempts < (n_per_genre * 3):
            emotion = random.choice(profile["emotion"])
            style = random.choice(profile["style"])
            instrument = random.choice(profile["instrument"])
            energy = random.choice(profile["energy"])
            texture = random.choice(TEXTURES)
            movement = random.choice(MOVEMENTS)

            # Format genre name to be readable (e.g., "children_story" -> "children story")
            readable_genre = genre.replace("_", " ")

            prompt = random.choice(TEMPLATES).format(
                emotion=emotion, genre=readable_genre, energy=energy,
                style=style, instrument=instrument, texture=texture, movement=movement
            )

            if prompt not in seen_prompts:
                seen_prompts.add(prompt)
                generated_results.append({
                    "genre": genre,
                    "prompt": prompt,
                    "emotion": emotion,
                    "tags": [genre, style, emotion, "instrumental", "background"]
                })
            attempts += 1

    return generated_results

# ==========================================
# 4. LOAD AI MUSIC MODEL
# ==========================================
print("\n[1/3] Loading MusicGen Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)
sampling_rate = model.config.audio_encoder.sampling_rate

# ==========================================
# 5. EXECUTION & INCREMENTAL SAVING
# ==========================================

TRACKS_PER_GENRE = 10 # 10 genres * 10 tracks = 100 tracks total
print(f"\n[2/3] Generating configurations for {TRACKS_PER_GENRE} tracks per genre...")
prompt_data_list = generate_genre_prompts(n_per_genre=TRACKS_PER_GENRE)
total_tracks = len(prompt_data_list)

# Setup Output Directory
drive_path = '/content/drive/MyDrive/Podcast_BGM_Library/'
os.makedirs(drive_path, exist_ok=True)
json_filepath = os.path.join(drive_path, "master_music_metadata.json")

# Load existing JSON if resuming a previous run
if os.path.exists(json_filepath):
    with open(json_filepath, 'r') as f:
        all_metadata = json.load(f)
else:
    all_metadata = []

start_index = len(all_metadata)

print(f"\n[3/3] Starting Batch Generation ({total_tracks} total tracks)...\n")

for i, data in enumerate(prompt_data_list[start_index:], start_index + 1):
    prompt_text = data["prompt"]
    genre = data["genre"]

    # Create an organized filename: e.g., 001_horror_bgm.wav
    filename = f"{i:03d}_{genre}_bgm.wav"

    print(f"--- Track {i}/{total_tracks} | Genre: {genre.upper()} ---")
    print(f"Prompt: {prompt_text}")

    # 1. Prepare inputs
    inputs = processor(
        text=[prompt_text],
        padding=True,
        return_tensors="pt",
    ).to(device)

    # 2. Generate (max_new_tokens=512 is ~10.2 seconds)
    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=512)

    # 3. Extract Audio Data
    audio_data = audio_values[0, 0].cpu().numpy()

    # 4. Save Locally and Copy to Drive
    local_filename = filename
    scipy.io.wavfile.write(local_filename, rate=sampling_rate, data=audio_data)
    shutil.copy(local_filename, os.path.join(drive_path, filename))
    print(f"✅ Saved: {filename}")

    # 5. Append Metadata and SAVE JSON IMMEDIATELY (Incremental Save)
    all_metadata.append({
        "id": i,
        "name": filename,
        "genre": genre,
        "prompt": prompt_text,
        "emotion": data["emotion"],
        "useful_tags": data["tags"]
    })

    with open(json_filepath, 'w') as f:
        json.dump(all_metadata, f, indent=4)

    print(f"💾 Master JSON updated.\n")

print(f"🎉 Complete! All {total_tracks} tracks and master metadata JSON are secured in: {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1/3] Loading MusicGen Model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[2/3] Generating configurations for 10 tracks per genre...

[3/3] Starting Batch Generation (100 total tracks)...

--- Track 1/100 | Genre: CHILDREN_STORY ---
Prompt: A joyful background music piece designed for children story narration, using woodwinds, with light and upbeat pacing, sparse and empty, with subtle variations, no vocals.
✅ Saved: 001_children_story_bgm.wav
💾 Master JSON updated.

--- Track 2/100 | Genre: CHILDREN_STORY ---
Prompt: A whimsical background score for a children story podcast, featuring glockenspiel, with medium intensity in a nursery style, soft and atmospheric, steady and consistent, no vocals.
✅ Saved: 002_children_story_bgm.wav
💾 Master JSON updated.

--- Track 3/100 | Genre: CHILDREN_STORY ---
Prompt: An instrumental acoustic track conveying a joyful mood for a children story story, driven by soft piano, light and upbeat energy, rich and layered, with subtle variations, no vocals.
✅ Saved: 003_children_story_bgm.wav
💾 Master JSON updated.

--- Track 4/1